# California Housing Price Prediction — Full Data Science Pipeline

**Dataset:** California Housing (1990 Census data)  
**Task:** Regression — predict median house prices  
**Models:** XGBoost Regressor vs. Artificial Neural Network (ANN)  

---

## Pipeline Overview
1. Data Collection & EDA
2. Preprocessing & Feature Engineering
3. Visualization & Correlation Analysis
4. Modeling (XGBoost + ANN)
5. Evaluation & Comparison (MSE, RMSE, MAE, R²)

---
## Step 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    r2_score, mean_absolute_percentage_error
)

import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('XGBoost version:', xgb.__version__)
print('TensorFlow version:', tf.__version__)
print('All libraries loaded successfully.')

---
## Step 2 — Data Collection & EDA

**California Housing Dataset** from 1990 U.S. Census

| Feature | Description |
|---------|-------------|
| MedInc | Median income (in tens of thousands) |
| HouseAge | Age of the house (years) |
| AveRooms | Average number of rooms |
| AveBedrms | Average number of bedrooms |
| Population | Population in the block |
| AveOccup | Average occupancy |
| Latitude | Geographic latitude |
| Longitude | Geographic longitude |
| **MedHouseValue** | **Median house price (target, in $100,000s)** |

In [ ]:
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseValue')
df = X.copy()
df['MedHouseValue'] = y

print(f'Dataset shape: {df.shape}')
print(f'\nFirst 10 rows:')
df.head(10)

In [ ]:
print('=== Dataset Info ===')
print(df.info())
print(f'\n=== Missing Values ===')
print(df.isnull().sum())
print(f'\n=== Duplicates ===')
print(f'{df.duplicated().sum()} duplicate rows')
print(f'\n=== Statistical Summary ===')
df.describe().round(3)

---
## Step 3 — Preprocessing

In [ ]:
# Remove duplicates
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Duplicates removed: {before - len(df)} rows')

# Detect and clip outliers using IQR method
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('MedHouseValue')

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower, upper)
    if n_out > 0:
        print(f'  Clipped {n_out} outliers in "{col}"')

print(f'\nFinal dataset shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')

---
## Step 4 — Feature Engineering

Create new meaningful features from existing ones:

| New Feature | Formula | Rationale |
|-------------|---------|----------|
| `RoomPerPerson` | `AveRooms / Population` | Space per capita |
| `BedroomRatio` | `AveBedrms / AveRooms` | Bedroom proportion |
| `PopulationDensity` | `Population / (Latitude-offset)` | Area density proxy |
| `IncomeToAge` | `MedInc / HouseAge` | Income growth proxy |

In [ ]:
# Room-to-population ratio
df['RoomPerPerson'] = (df['AveRooms'] / (df['Population'] + 1)).round(4)

# Bedroom to room ratio
df['BedroomRatio'] = (df['AveBedrms'] / (df['AveRooms'] + 1)).round(4)

# Population density (using latitude as proxy for area)
df['PopDensity'] = (df['Population'] / (np.abs(df['Latitude']) + 1)).round(2)

# Income-to-age ratio (economic growth indicator)
df['IncomeToAge'] = (df['MedInc'] / (df['HouseAge'] + 1)).round(4)

print('New features added:', ['RoomPerPerson', 'BedroomRatio', 'PopDensity', 'IncomeToAge'])
print(f'Dataset shape after feature engineering: {df.shape}')

---
## Step 5 — Visualization

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['MedHouseValue'], bins=50, color='#2196F3', edgecolor='white', linewidth=1)
axes[0].set_title('Distribution of Median House Values', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price ($100,000s)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['MedHouseValue'], vert=True)
axes[1].set_title('Boxplot of House Prices', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Price ($100,000s)')

print(f'Price range: ${df["MedHouseValue"].min():.2f}M - ${df["MedHouseValue"].max():.2f}M')
print(f'Mean price: ${df["MedHouseValue"].mean():.3f}M')
print(f'Median price: ${df["MedHouseValue"].median():.3f}M')

plt.tight_layout()
plt.savefig('plot_target_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# Scatter plots: Top predictive features vs target
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features_to_plot = ['MedInc', 'HouseAge', 'AveRooms', 'Population']

for ax, feat in zip(axes.flat, features_to_plot):
    ax.scatter(df[feat], df['MedHouseValue'], alpha=0.4, s=20, color='#2196F3', edgecolors='none')
    ax.set_xlabel(feat, fontweight='bold')
    ax.set_ylabel('MedHouseValue', fontweight='bold')
    ax.set_title(f'{feat} vs House Price', fontsize=11)

plt.suptitle('Feature Relationships with Target', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr = df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    annot_kws={'size': 8}, square=True
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plot_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
feat_list = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']

for ax, feat in zip(axes.flat, feat_list):
    ax.hist(df[feat], bins=40, color='#42A5F5', edgecolor='white', linewidth=0.8)
    ax.set_title(feat, fontweight='bold')
    ax.set_ylabel('Frequency')

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Geographic visualization
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    df['Longitude'], df['Latitude'],
    c=df['MedHouseValue'], cmap='viridis',
    alpha=0.7, s=30, edgecolors='none'
)
cbar = plt.colorbar(scatter)
cbar.set_label('House Price ($100,000s)', fontsize=11)
plt.xlabel('Longitude', fontsize=12, fontweight='bold')
plt.ylabel('Latitude', fontsize=12, fontweight='bold')
plt.title('California Housing Prices by Geographic Location', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_geographic.png', bbox_inches='tight')
plt.show()

---
## Step 6 — Modeling

### 6.1 — Prepare Train/Test Split

In [ ]:
X = df.drop('MedHouseValue', axis=1)
y = df['MedHouseValue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Scale features for ANN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Target range: [{y.min():.2f}, {y.max():.2f}]')

### 6.2 — Model 1: XGBoost Regressor

XGBoost (Extreme Gradient Boosting) is an ensemble method that builds trees sequentially, each correcting errors from previous trees.

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    early_stopping_rounds=50,
    verbose=False
)

xgb_preds = xgb_model.predict(X_test)

print('XGBoost trained successfully.')
print(f'Best iteration: {xgb_model.best_iteration}')

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 7))
colors = ['#F44336' if x > importance_df['Importance'].median() else '#90CAF9' 
          for x in importance_df['Importance']]
plt.barh(importance_df['Feature'], importance_df['Importance'], color=colors, edgecolor='white')
plt.title('XGBoost — Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('plot_xgb_importance.png', bbox_inches='tight')
plt.show()

### 6.3 — Model 2: Artificial Neural Network (ANN)

A feedforward ANN with:
- **Input layer**: 12 features (8 original + 4 engineered)
- **3 hidden layers** with ReLU + Batch Normalization + Dropout
- **Output layer**: 1 neuron (linear activation for regression)

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

n_features = X_train_scaled.shape[1]

ann = keras.Sequential([
    layers.Input(shape=(n_features,)),
    
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.1),
    
    layers.Dense(1)  # Linear output for regression
], name='ANN_Housing')

ann.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

ann.summary()

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=20, restore_best_weights=True
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=10, min_lr=1e-5
)

history = ann.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)

print(f'Training stopped at epoch {len(history.history["loss"])}')
print(f'Final val MAE: {history.history["val_mae"][-1]:.4f}')

In [ ]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train MSE', color='#2196F3')
axes[0].plot(history.history['val_loss'], label='Val MSE', color='#F44336')
axes[0].set_title('ANN — Mean Squared Error', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].set_yscale('log')

axes[1].plot(history.history['mae'], label='Train MAE', color='#2196F3')
axes[1].plot(history.history['val_mae'], label='Val MAE', color='#F44336')
axes[1].set_title('ANN — Mean Absolute Error', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_ann_history.png', bbox_inches='tight')
plt.show()

In [ ]:
ann_preds = ann.predict(X_test_scaled, verbose=0).flatten()

print('ANN predictions generated.')
print(f'Prediction range: [{ann_preds.min():.2f}, {ann_preds.max():.2f}]')

---
## Step 7 — Evaluation & Comparison

In [ ]:
# Residuals analysis
xgb_residuals = y_test - xgb_preds
ann_residuals = y_test - ann_preds

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# XGBoost residuals
axes[0, 0].scatter(xgb_preds, xgb_residuals, alpha=0.4, s=20, color='#2196F3')
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0, 0].set_title('XGBoost — Residuals', fontweight='bold')
axes[0, 0].set_xlabel('Predicted Value')
axes[0, 0].set_ylabel('Residual')

# ANN residuals
axes[0, 1].scatter(ann_preds, ann_residuals, alpha=0.4, s=20, color='#F44336')
axes[0, 1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0, 1].set_title('ANN — Residuals', fontweight='bold')
axes[0, 1].set_xlabel('Predicted Value')
axes[0, 1].set_ylabel('Residual')

# XGBoost residuals histogram
axes[1, 0].hist(xgb_residuals, bins=40, color='#2196F3', edgecolor='white', alpha=0.7)
axes[1, 0].set_title('XGBoost — Residuals Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Residual')
axes[1, 0].set_ylabel('Frequency')

# ANN residuals histogram
axes[1, 1].hist(ann_residuals, bins=40, color='#F44336', edgecolor='white', alpha=0.7)
axes[1, 1].set_title('ANN — Residuals Distribution', fontweight='bold')
axes[1, 1].set_xlabel('Residual')
axes[1, 1].set_ylabel('Frequency')

plt.suptitle('Residual Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_residuals.png', bbox_inches='tight')
plt.show()

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# XGBoost
axes[0].scatter(y_test, xgb_preds, alpha=0.4, s=20, color='#2196F3')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_title('XGBoost — Actual vs Predicted', fontweight='bold')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].legend()

# ANN
axes[1].scatter(y_test, ann_preds, alpha=0.4, s=20, color='#F44336')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[1].set_title('ANN — Actual vs Predicted', fontweight='bold')
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_pred_vs_actual.png', bbox_inches='tight')
plt.show()

In [ ]:
# Calculate metrics
def get_regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    return {
        'MSE': round(mse, 6),
        'RMSE': round(rmse, 4),
        'MAE': round(mae, 4),
        'R2 Score': round(r2, 4),
        'MAPE (%)': round(mape * 100, 2)
    }

xgb_metrics = get_regression_metrics(y_test, xgb_preds)
ann_metrics = get_regression_metrics(y_test, ann_preds)

comparison_df = pd.DataFrame([
    xgb_metrics,
    ann_metrics
], index=['XGBoost', 'ANN'])

print('\n' + '='*70)
print('           REGRESSION MODEL PERFORMANCE COMPARISON')
print('='*70)
print(comparison_df.to_string())
print('='*70)

In [ ]:
# Metrics comparison bar chart
metrics_names = ['MSE', 'RMSE', 'MAE', 'R2 Score']
xgb_vals = [xgb_metrics['MSE'], xgb_metrics['RMSE'], xgb_metrics['MAE'], xgb_metrics['R2 Score']]
ann_vals = [ann_metrics['MSE'], ann_metrics['RMSE'], ann_metrics['MAE'], ann_metrics['R2 Score']]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Normalize values for visualization (except R2)
metrics_data = [
    (['XGBoost', 'ANN'], [xgb_metrics['MSE'], ann_metrics['MSE']], 'MSE (lower is better)', '#FFC107'),
    (['XGBoost', 'ANN'], [xgb_metrics['RMSE'], ann_metrics['RMSE']], 'RMSE (lower is better)', '#FF9800'),
    (['XGBoost', 'ANN'], [xgb_metrics['MAE'], ann_metrics['MAE']], 'MAE (lower is better)', '#2196F3'),
    (['XGBoost', 'ANN'], [xgb_metrics['R2 Score'], ann_metrics['R2 Score']], 'R2 Score (higher is better)', '#4CAF50')
]

for ax, (labels, values, title, color) in zip(axes.flat, metrics_data):
    bars = ax.bar(labels, values, color=color, edgecolor='white', linewidth=2)
    ax.set_title(title, fontweight='bold', fontsize=11)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Regression Metrics Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_metrics_comparison.png', bbox_inches='tight')
plt.show()

---
## Step 8 — Final Summary & Conclusions

In [ ]:
print('\n' + '='*75)
print('                    FINAL PROJECT SUMMARY')
print('='*75)
print(f'Dataset     : California Housing (1990 Census)')
print(f'Task        : Regression - Predict Median House Prices')
print(f'Samples     : {len(df)} (after preprocessing)')
print(f'Features    : {X.shape[1]} (including 4 engineered features)')
print(f'Train / Test: 80% / 20%')
print()
print('-'*75)
print(f"{'Metric':<20} {'XGBoost':>20} {'ANN':>20}  {'Winner':>10}")
print('-'*75)

metrics_list = ['MSE', 'RMSE', 'MAE', 'R2 Score']
for metric in metrics_list:
    xgb_v = xgb_metrics[metric]
    ann_v = ann_metrics[metric]
    
    if metric == 'R2 Score':
        winner = '[XGB]' if xgb_v > ann_v else ('[TIE]' if xgb_v == ann_v else '[ANN]')
    else:
        winner = '[XGB]' if xgb_v < ann_v else ('[TIE]' if xgb_v == ann_v else '[ANN]')
    
    print(f'{metric:<20} {xgb_v:>20.6f} {ann_v:>20.6f}  {winner:>10}')

print('-'*75)
print(f"{'MAPE (%)':<20} {xgb_metrics['MAPE (%)']:>20.2f} {ann_metrics['MAPE (%)']:>20.2f}")
print('='*75)

best = 'XGBoost' if xgb_metrics['R2 Score'] > ann_metrics['R2 Score'] else 'ANN'
print(f'\n  Best overall model: >> {best} <<')
print('='*75)

---
## Conclusions

### Dataset Insights
- Housing prices range from $0.15M to $5.0M (in $100,000 units)
- **Median Income (`MedInc`)** is the strongest predictor of house prices (highest correlation with target)
- Geographic location matters: coastal properties (high latitude/longitude variance) show price variation
- House age and population density also contribute significantly to price prediction
- The dataset exhibits heteroscedasticity (prediction errors increase with price magnitude)

### Model Comparison

| Aspect | XGBoost | ANN |
|--------|---------|-----|
| **Speed** | Trains quickly (~500 iterations) | Slower (~150 epochs) |
| **Interpretability** | High (feature importance scores) | Low (black box) |
| **Robustness** | Handles outliers well | Sensitive to outliers |
| **Non-linear patterns** | Excellent | Excellent |
| **Feature scaling needed** | No | Yes (critical) |

### Key Metrics Explained
- **MSE**: Mean squared error — penalizes large errors heavily
- **RMSE**: Root MSE — interpretable in same units as target
- **MAE**: Mean absolute error — average absolute prediction error
- **R² Score**: Fraction of variance explained (0-1, higher is better)
- **MAPE**: Mean absolute percentage error — relative error across price ranges

### Best Model
The comparison shows which model generalizes better on unseen data. **XGBoost** typically excels on tabular data with gradient boosting's sequential error correction, while **ANN** can capture complex feature interactions with sufficient training data.